In [1]:
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import sklearn
import sklearn.ensemble as ske
import time
import numpy as np
import pynq
import math
import datetime
import struct
import asyncio
import os
import re
import ast
import sys
import pickle
%cd /home/xilinx/pynq/Moyogi
from converter import Converter
from MultiDepthRandomForest import MultiDepthRandomForestClassifier
from Dataset import import_accelerometer_pca, import_satellite_pca, import_vowel_pca

/usr/local/share/pynq-venv/lib/python3.10/site-packages/pynq/Moyogi


In [2]:
class NOInst:
    def __init__(self, nodeRA, attr_id, threshold, lctype, lcinfo, rctype, rcinfo, is_valid = True, depth_indicator = 0):
        self.nodeRA = nodeRA
        self.attr_id = attr_id
        self.threshold = threshold
        self.lctype = lctype
        self.lcinfo = lcinfo
        self.rctype = rctype
        self.rcinfo = rcinfo
        self.is_valid = is_valid 
        self.depth_indicator = depth_indicator
        
def transform_into_instructions(DTs, node_instr_dict, path_index, max_depth, min_depth):
    
    for i in range(max_depth):
        node_instr_dict[max_depth*path_index+i] = [] #initialize the list of instructions related to BRAM i to an empty list
    
    indexes_dict = {}
    for i in range(max_depth):
        indexes_dict[i] = 0
    
    for i, DT in enumerate(DTs):
        this_level_nodes = []
        next_level_nodes = [0]
        for j in range(DT.tree_.max_depth):
            this_level_nodes = next_level_nodes.copy()
            next_level_nodes = []
            if j < DT.tree_.max_depth-1:
                counter = indexes_dict[j+1]
                
            for k,node in enumerate(this_level_nodes):
                left_child = DT.tree_.children_left[node]
                right_child = DT.tree_.children_right[node]
                attr_id = DT.tree_.feature[node]
                threshold = DT.tree_.threshold[node]
                
                if DT.tree_.feature[left_child] >= 0:
                    next_level_nodes.append(left_child)
                    lctype = True
                    lcinfo = counter
                    counter += 1
                else:
                    lctype = False
                    lcinfo = np.argmax(DT.tree_.value[left_child,0])
                    
                if DT.tree_.feature[right_child] >= 0:
                    next_level_nodes.append(right_child)
                    rctype = True
                    rcinfo = counter
                    counter += 1
                else:
                    rctype = False
                    rcinfo = np.argmax(DT.tree_.value[right_child,0])
                
                node_instr_dict[max_depth*path_index+j].append(NOInst(indexes_dict[j],attr_id, threshold, lctype, lcinfo, rctype, rcinfo, True, DT.tree_.max_depth - min_depth))
                indexes_dict[j] += 1

In [9]:
class Accel:

    def __init__(self, overlay, converter, platform='Ultra96_v2', n_brams=100, n_samples=1000, dim = 1000, n_classes=3, n_attr=5, n_weights=5, trees_in_bram=3, max_inst=576, frq_MHZ = 100):
        self.overlay = overlay
        self.dim = dim
        self.n_samples = n_samples
        self.dma = overlay.axi_dma_0
        self.brams = []
        self.brams_bit = int(np.math.ceil(np.log2(n_brams)))
        self.nodera_bit = int(np.math.ceil(np.log2(max_inst)))
        self.info_bit = int(max(np.math.ceil(np.log2(n_classes)),self.nodera_bit))
        self.attr_bit = int(np.math.ceil(np.log2(n_attr)))
        self.tree_bit = int(np.math.ceil(np.log2(trees_in_bram)))
        self.scores_bit = int(np.math.ceil(np.log2(n_brams*trees_in_bram)))
        self.depths_bit = int(np.math.ceil(np.log2(n_weights)))
        self.n_attr = n_attr
        self.n_classes = n_classes
        self.n_weights = n_weights
        for i in range(n_brams):
            variable_value = getattr(self.overlay,"axi_bram_ctrl_" + str(i))
            self.brams.append(variable_value)
            
        self.converter = converter
        self.frequency = frq_MHZ*(10**6)

        self.buff_in = pynq.allocate((self.n_samples,self.dim),dtype=np.uint8)
        self.buff_in[:] = np.zeros((self.n_samples,self.dim),dtype=np.uint8)
        self.buff_out = pynq.allocate((self.n_samples,self.dim),dtype=np.uint8)
        self.buff_out[:] = np.zeros((self.n_samples,self.dim),dtype=np.uint8)
        self.platform = platform
    
    def init_bram(self,bram,node_instructions):
        total_bits = 32 + self.attr_bit + 2*self.info_bit + 3 + self.depths_bit
        if total_bits <= 64:
            for i,node_instr in enumerate(node_instructions):  
                uint_result = self.converter.forward_conversion(input_data=node_instr.threshold, signed=True, total_bits=16, fractional_bits=8)
                
                #uint_result = 0
                    
                first_bytes =  int(uint_result) | 0 << 16
                
                #print(type(node_instr.rctype),type(node_instr.lctype),type(node_instr.rcinfo),type(node_instr.lcinfo),type(node_instr.attr_id),type(node_instr.nodeRA))
                #print("depth_indicator: shift " + str((self.attr_bit + 2*self.info_bit + 3)) + ", data=" + str(node_instr.depth_indicator))
                #print("is_valid: shift " + str((self.attr_bit + 2*self.info_bit + 2)) + ", data=" + str(node_instr.is_valid))
                #print("rctype: shift " + str((self.attr_bit + 2*self.info_bit + 1)) + ", data=" + str(node_instr.rctype))
                #print("lctype: shift " + str(self.attr_bit + 2*self.info_bit) + ", data=" + str(node_instr.lctype))
                #print("rcinfo: shift " + str(self.attr_bit + self.info_bit) + ", data=" + str(node_instr.rcinfo))
                #print("lcinfo: shift " + str(self.attr_bit) + ", data=" + str(node_instr.lcinfo))
                #print("attr id: " + str(node_instr.attr_id) + " shift 0")
                
                second_bytes = int((node_instr.depth_indicator << (self.attr_bit + 2*self.info_bit + 3)) | \
                              (node_instr.is_valid << (self.attr_bit + 2*self.info_bit + 2)) | \
                              (node_instr.rctype << (self.attr_bit + 2*self.info_bit + 1)) | \
                              (node_instr.lctype << (self.attr_bit + 2*self.info_bit)) | \
                              (node_instr.rcinfo << (self.attr_bit + self.info_bit)) | \
                              (node_instr.lcinfo << (self.attr_bit) ) | node_instr.attr_id)
                
                bram.write(node_instr.nodeRA*8, first_bytes)
                bram.write(node_instr.nodeRA*8 + 4, second_bytes)
                #print("Writing " + str(format(second_byte,'032b')) + " at " + str(node_instr.nodeRA*8))
                #print("Writing " + str(format(second_byte,'032b')) + " at " + str(node_instr.nodeRA*8 + 4))
                #print("NOINST: ", first_byte | (second_byte << 32))
                #print(first_byte, second_byte)
                
                '''
                #old version
                first_byte = int(uint_result)
                second_byte = (node_instr.rctype << self.nodera_bit + self.attr_bit + 2*self.info_bit + self.brams_bit) | \
                              (node_instr.lctype << self.nodera_bit + self.attr_bit + 2*self.info_bit) | \
                              (node_instr.rcinfo << self.nodera_bit + self.attr_bit + self.info_bit) | \
                              (node_instr.lcinfo << self.nodera_bit + self.attr_bit ) | \
                              (node_instr.attr_id << self.nodera_bit) | node_instr.nodeRA
                bram.write(node_instr.nodeRA*8, first_byte)
                bram.write(node_instr.nodeRA*8 + 4, second_byte)
                '''
                
        else:
            print("ERRORE")
            for i,node_instr in enumerate(node_instructions):  
                uint_result = self.converter.forward_conversion(input_data=node_instr.threshold, signed=True, total_bits=32, fractional_bits=16)
                first_byte = struct.pack('I',uint_result)
                bram.write(i*16, first_byte)
                bits = [self.nodera_bit,self.attr_bit,self.info_bit,self.info_bit,self.brams_bit,self.brams_bit,0]
                values = [node_instr.nodeRA,node_instr.attr_id,node_instr.lcinfo,node_instr.rcinfo,node_instr.lctype,node_instr.rctype]
                j=0
                second_byte = 0
                while np.sum(bits[:j+1])<=32:
                    second_byte = second_byte | (values[j] << int(np.sum(bits[:j])))
                    j += 1
                to_fill = 32 - np.sum(bits[:j])
                binary_value = bin(values[j])[2:].zfill(bits[j])
                second_byte = second_byte | (int(binary_value[-to_fill:],2) << int(np.sum(bits[:j])))
                bram.write(i*16 + 4, second_byte)
                third_byte = int(binary_value[:-to_fill],2)
                j += 1
                while(j<6):
                    third_byte = third_byte | (values[j] << int(np.sum(bits[:j]) - 32))
                    j += 1
                bram.write(i*16 + 8, third_byte)
        
    def init_samples(self,samples):
        assert self.n_samples == len(samples)
        for s in tqdm(range(self.n_samples)):
            for i in range(len(samples[s].features)):
                int_value = self.converter.forward_conversion(input_data=samples[s].features[i], signed=True, total_bits=32, fractional_bits=16)
                b = struct.pack('I', int_value)
                a = list(b)
                a = [int(x) for x in a]
                for j,value in enumerate(a):
                    self.buff_in[s,(4*i+j)] = value
            self.buff_in[s,4*len(samples[s].features)+1] = samples[s].offset
            self.buff_in[s,4*len(samples[s].features)+2] = samples[s].shift
            self.buff_in[s,4*len(samples[s].features)+3] = samples[s].search_for_root
            self.buff_in[s,4*len(samples[s].features)+4] = samples[s].tree_to_exec
            
            for i in range(len(samples[s].scores)):
                self.buff_in[s,4*len(samples[s].features)+6+2*i] = samples[s].scores[i]
                
            for i in range(len(samples[s].weights)):
                int_value = self.converter.forward_conversion(input_data=samples[s].weights[i], signed=True, total_bits=16, fractional_bits=6)
                b = struct.pack('I', int_value)
                a = list(b)
                a = [int(x) for x in a]
                for j,value in enumerate(a[:2]):
                    self.buff_in[s,4*len(samples[s].features)+5+2*len(samples[s].scores)+2*i+j+1] = value
        print(self.buff_in[0:2])
    
        
    def init_accel(self,samples,node_instructions_dict):
        
        for i,bram in tqdm(enumerate(self.brams)):
            self.init_bram(bram,node_instructions_dict[i])
        
        self.init_samples(samples)
        
    def exec_test(self):
        
        '''
        n_samples_per_batch = 130
            
        stop = False
        i = 0
        while not stop:
            start = n_samples_per_batch*i
            if n_samples_per_batch*(i+1) >= self.n_samples:
                end = self.n_samples
                stop = True
            else:
                end = n_samples_per_batch*(i+1)
                
            self.dma.sendchannel.transfer(self.buff_in[start:end,:])
            self.dma.recvchannel.transfer(self.buff_out[start:end,:])
            
            if i==0:
                self.dma.sendchannel.start()
                self.dma.recvchannel.start()
                
            print("waiting for send to finish")
            self.dma.sendchannel.wait()
            print("waiting for receive to finish")
            self.dma.recvchannel.wait()
            
            i += 1
        '''
        
        self.dma.sendchannel.transfer(self.buff_in)
        self.dma.recvchannel.transfer(self.buff_out)
        
        self.dma.sendchannel.start()
        self.dma.recvchannel.start()
        
        print("waiting...")
        
        self.dma.recvchannel.wait()
        
        self.buff_out.invalidate()
        outs = self.buff_out
        samples = []
        clock_cycles = []
        for out in outs:
            features = []
            for i in range(self.n_attr):
                int_result = out[i*4]*(2**0)+out[i*4+1]*(2**8)+out[i*4+2]*(2**16)+out[i*4+3]*(2**24)
                features.append(self.converter.backward_conversion(input_data=int_result, total_bits=32, fractional_bits=16)[0])
            scores = []
            for i in range(self.n_classes):
                int_result = out[4*self.n_attr+6+2*i]*(2**0)+out[4*self.n_attr+6+2*i+1]*(2**8)
                scores.append(self.converter.backward_conversion(input_data=int_result, total_bits=16, fractional_bits=6)[0])
            weights = np.zeros(self.n_weights)
            
            cycles = 0
            for i in range(4):
                cycles += out[4*self.n_attr+6+2*self.n_classes+i]*2**(8*i)
            
            sample = Sample_w_Score(features,out[4*self.n_attr+2],out[4*self.n_attr],out[4*self.n_attr+3],out[4*self.n_attr+4],scores, weights)
            samples.append(sample)
            clock_cycles.append(cycles)
        #elapsed_time = (last_out-first_in)*10e-9
        #latency = (last_out-last_in)*10e-9
        #troughput = self.buff_out.shape[0]/((last_out-first_out)*10e-9)
        throughput = self.frequency*self.n_samples/((clock_cycles[-1]-clock_cycles[0]))
        latencies = []
        for i,cc in enumerate(clock_cycles[:-1]):
            latencies.append(cc-3*i)
            
        avg_latency = np.mean(latencies)
        
        del self.buff_out
        del self.buff_in
        return samples, throughput, avg_latency, outs, clock_cycles

In [10]:
class Sample_w_Score:
    def __init__(self, features , shift, offset, search_for_root, tree_to_exec, scores, weights):
        self.features = features
        self.shift = shift
        self.offset = offset
        self.search_for_root = search_for_root
        self.tree_to_exec = tree_to_exec
        self.scores = scores
        self.weights = weights

In [34]:
def main(X_test, overlay, total_pes, node_instructions_dict, n_attr, n_classes, max_trees_per_set, max_inst, frq, n_depths = 5, width = None):
    
    seed = 1234
    if width is None:
        dim = 4*n_attr + 2*(n_classes + n_depths) + 6
    else:
        dim = width
        
    while(dim%4 != 0):
        dim += 1
    
    print(dim)
    np.random.seed(seed)
    shift = 0
    offset = 0
    search_for_root = 1
    tree_to_exec = 0
    scores = np.zeros(n_classes)
    instances = []
    weights = np.ones(n_depths)/4
    n_samples = 130#X_test.shape[0]
    latencies = []
    for j in range(15):
        instances = []
        for i in range(n_samples):
            index = np.random.randint(X_test.shape[0])
            sample = Sample_w_Score(X_test[index], shift, 0, search_for_root, 0, scores, weights)
            instances.append(sample)

        converter = Converter()

        '''
        throughputs = []
        latencies = []
        skip = False
        final = 0
        while(not skip):
            test = pynq.Overlay(overlay,download=True)
            accel = Accel(test, converter, n_brams = total_pes, n_samples = len(instances), dim = dim, n_classes = n_classes, n_weights = len(weights), trees_in_bram = max_trees_per_set, n_attr = n_attr, max_inst = max_inst,frq_MHZ = frq)
            accel.init_accel(instances,node_instructions_dict)
            samples, throughput, latency, buff, clock_cycles = accel.exec_test()
            print(latency)
            print(clock_cycles)
            if latencies.count(latency) >= 1:
                skip = True
                final = latency

            throughputs.append(throughput)
            latencies.append(latency)
        '''
        
        test = pynq.Overlay(overlay,download=True)
        accel = Accel(test, converter, n_brams = total_pes, n_samples = len(instances), dim = dim, n_classes = n_classes, n_weights = len(weights), trees_in_bram = max_trees_per_set, n_attr = n_attr, max_inst = max_inst,frq_MHZ = frq)
        accel.init_accel(instances,node_instructions_dict)
        samples, throughput, latency, buff, clock_cycles = accel.exec_test()
        latencies.append(latency)
        print(latency)
        print(clock_cycles)

    
    print("correct average latency found")
    lat = np.mean(latencies)/(166*1e6)
    print(lat)
        
    return samples, lat, clock_cycles 

In [35]:
root_folder = './DeploysRescheduling/new/'
folders = os.listdir(root_folder)
folders.sort()
folders

['accelerometer_1080_5',
 'accelerometer_180_7',
 'accelerometer_30_9',
 'satellite_180_7',
 'satellite_30_9',
 'satellite_900_5',
 'vowel_135_7',
 'vowel_20_9',
 'vowel_720_5']

In [36]:
samples_list = {}
latencies = {}
clock_cycles_list = {}
ordering_dirs_list = {}
for folder in folders:
    print(f"Running {folder}")
    
    elements = os.listdir(root_folder + folder)
    for elem in elements:
        if elem.startswith("Deploy"):
            deploy_dir = elem
    overlay = root_folder + folder + '/' + deploy_dir + '/design_2_wrapper.bit'
    #print("Executing project: " + folder)
    folderParams = re.findall(r'\d+', deploy_dir)
    n_trees = int(folderParams[1])
    max_depth = int(folderParams[0])
    frq = int(folderParams[2])
    n_paths = int(folderParams[3])
    n_attr = int(folderParams[4])
    min_depth = max_depth - 4
    n_depths = max_depth - min_depth + 1
    %cd data
    if folder.startswith("accelerometer"):
        X_train, X_val, X_test, Y_train, Y_val, Y_test = import_accelerometer_pca()
        n_classes = 7
        width = int(384/8)
    elif folder.startswith("satellite"):
        X_train, X_val, X_test, Y_train, Y_val, Y_test = import_satellite_pca()
        n_classes = 6
        width = int(448/8)
    elif folder.startswith("vowel"):
        X_train, X_val, X_test, Y_train, Y_val, Y_test = import_vowel_pca()
        n_classes = 11
        width = int(640/8)
    else:
        print("Dataset not recognized")
    
    %cd ..
    #print("selected number of trees", n_trees)
    if n_trees < n_paths*n_depths:
        updated_n_trees = n_paths*n_depths
    else:
        updated_n_trees = n_trees
        
    #print("trained number of trees", updated_n_trees)
    
    bram_size = 36*1024
    instruction_per_bram = int(bram_size/64)
    trees_per_path = int(updated_n_trees/n_paths) #assume at the moment that the total number of trees is multiple of n_paths
    trees_per_path_per_depth = int(trees_per_path/n_depths)
    max_trees_per_set = instruction_per_bram//((2**(max_depth-1)))*n_depths
    necessary_set_of_pes = int(math.ceil(updated_n_trees/max_trees_per_set))
    
    total_pes = max_depth*max(necessary_set_of_pes,n_paths)
    ordering_dirs = os.listdir(root_folder + folder)
    samples_list[folder] = []
    latencies[folder] = []
    clock_cycles_list[folder] = []
    ordering_dirs_list[folder] = []
    for ordering_dir in ordering_dirs:
        if ordering_dir.startswith("Deploy"):
            continue
        print(f"running with order: {ordering_dir}")
        DT_paths = os.listdir(root_folder + folder + "/" + ordering_dir) #list DT .pkl files
        DTs = {}
        for i in range(max(necessary_set_of_pes,n_paths)): #create a dict that maps each set to an empty list of DTs
            DTs[i] = []
        
        for i in range(len(DT_paths)):
            with open(root_folder + folder + "/" + ordering_dir + "/" + DT_paths[i], 'rb') as file:
                model = pickle.load(file)
            
            DTs[i%(max(necessary_set_of_pes,n_paths))].append(model) #add the DTs in order one for each path
        
        node_instr_dict = {}
        for i in range(max(necessary_set_of_pes,n_paths)):
            transform_into_instructions(DTs[i], node_instr_dict, i, max_depth, min_depth)
        
        samples, latency, clock_cycles = main(X_test, overlay, total_pes, node_instr_dict, n_attr, n_classes, max_trees_per_set, instruction_per_bram, frq, n_depths, width)
        samples_list[folder].append(samples)
        latencies[folder].append(latency)
        ordering_dirs_list[folder].append(ordering_dir)
        clock_cycles_list[folder].append(clock_cycles)
    

Running accelerometer_1080_5
/usr/local/share/pynq-venv/lib/python3.10/site-packages/pynq/Moyogi/data
/usr/local/share/pynq-venv/lib/python3.10/site-packages/pynq/Moyogi
running with order: accelerometer_nTree_1080_depth_1_5_resch_False_autoencoder_False
48


0it [00:00, ?it/s]

  0%|          | 0/130 [00:00<?, ?it/s]

[[219  53 192 255  60 150 221 255 145 251   2   0 130 105  28   0   0   0
    0   1   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   16   0  16   0  16   0  16   0  16   0   0   0]
 [ 71 116 187 255 107   0 214 255 176  97  23   0  97 139 220 255   0   0
    0   1   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   16   0  16   0  16   0  16   0  16   0   0   0]]
waiting...
14340.899224806202
[11737, 11797, 11849, 12008, 12069, 12145, 12149, 12195, 12200, 12202, 12282, 12305, 12310, 12473, 12479, 12500, 12519, 12526, 12737, 12786, 12819, 12996, 13011, 13057, 13108, 13292, 13313, 13321, 13451, 13476, 13479, 13568, 13646, 13666, 13667, 13707, 13740, 13768, 13773, 13844, 13848, 13950, 13986, 14080, 14120, 14124, 14179, 14225, 14235, 14277, 14292, 14332, 14373, 14403, 14415, 14427, 14449, 14482, 14526, 14541, 14544, 14677, 14693, 14740, 14741, 14744, 14747, 14882, 14884, 14935, 14942, 14991, 14996, 15009, 15027, 15053, 15059, 15115, 15118, 15203, 15204, 152

0it [00:00, ?it/s]

...

  0%|          | 0/130 [00:00<?, ?it/s]

[[185 205 239 255  83 208 196 255 139 111  23   0  72 232  11   0 144  36
  237 255 235 123  19   0 161  96   9   0 228 172   9   0   0   0   0   1
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0  16   0  16   0  16   0  16   0  16   0   0   0
    0   0   0   0   0   0   0   0]
 [129  65  62   0   4  31  27   0 163 232  82   0 129 250 232 255 172  49
   16   0  67 114   9   0  36 176   3   0  12 228  12   0   0   0   0   1
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0  16   0  16   0  16   0  16   0  16   0   0   0
    0   0   0   0   0   0   0   0]]
waiting...
20497.077519379844
[15959, 16680, 17039, 17248, 17276, 17623, 17678, 17775, 17846, 18190, 18457, 18640, 18676, 18777, 18942, 19105, 19189, 19314, 19387, 19400, 19435, 19454, 19466, 19546, 19555, 19558, 19580, 19587, 19890, 19965, 20138, 20219, 20287, 20301, 20359, 20390, 20404, 20416, 20439, 20554, 20580, 20620, 20727, 20777

0it [00:00, ?it/s]

  0%|          | 0/130 [00:00<?, ?it/s]

[[ 20  67 235 255  22 202  15   0 168  84  41   0 140 120  31   0  57 151
   72   0  94 242 212 255 105  86 255 255 224 109  18   0   0   0   0   1
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0  16   0  16   0  16   0  16   0  16   0   0   0
    0   0   0   0   0   0   0   0]
 [ 77 128  26   0  29 126   2   0 243 157  42   0 241 100 235 255  30  64
   58   0 191 131 255 255 123  35 240 255 216 233  14   0   0   0   0   1
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0  16   0  16   0  16   0  16   0  16   0   0   0
    0   0   0   0   0   0   0   0]]
waiting...
20215.5503875969
[16347, 16867, 16890, 17159, 17339, 17370, 17607, 17781, 17787, 17939, 18388, 18454, 18553, 18564, 18606, 18608, 18988, 19003, 19031, 19106, 19134, 19176, 19189, 19193, 19233, 19277, 19347, 19462, 19518, 19522, 19685, 19729, 19759, 19786, 19833, 20040, 20123, 20130, 20270, 20272, 20301, 20304, 20334, 20400, 

In [37]:
for key in latencies.keys():
    for j in range(4):
        print(ordering_dirs_list[key][j], latencies[key][j])

accelerometer_nTree_1080_depth_1_5_resch_False_autoencoder_False 8.575208430621712e-05
accelerometer_nTree_1080_depth_1_5_resch_True_autoencoder_True_method_Autoencoder 8.507184396500732e-05
accelerometer_nTree_1080_depth_1_5_resch_True_autoencoder_False_method_Siamese 8.573502381619499e-05
accelerometer_nTree_1080_depth_1_5_resch_True_autoencoder_False_method_Hungarian 8.53568755642726e-05
accelerometer_nTree_180_depth_3_7_resch_True_autoencoder_False_method_Hungarian 2.0174589832197004e-05
accelerometer_nTree_180_depth_3_7_resch_False_autoencoder_False 2.005753556863112e-05
accelerometer_nTree_180_depth_3_7_resch_True_autoencoder_True_method_Autoencoder 2.003253946016625e-05
accelerometer_nTree_180_depth_3_7_resch_True_autoencoder_False_method_Siamese 2.018088789265589e-05
accelerometer_nTree_30_depth_5_9_resch_True_autoencoder_False_method_Siamese 3.0719560412191406e-06
accelerometer_nTree_30_depth_5_9_resch_True_autoencoder_True_method_Autoencoder 2.964920145699075e-06
acceleromete